In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix, recall_score, balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder

# ----------------------------
# Reproducibility
# ----------------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# ----------------------------
# Config
# ----------------------------
columns_to_discard = [
    'collision_index',
    'collision_ref_no',
    'location_easting_osgr',
    'location_northing_osgr',
    'local_authority_ons_district',
    'local_authority_highway',
    'local_authority_highway_current',
    'lsoa_of_accident_location',
    'AREFNO',
    'collision_severity',
    'enhanced_severity_collision',
    'collision_injury_based',
    'collision_adjusted_severity_serious',
    'collision_adjusted_severity_slight',
    'CREFNO',
    'latitude',
    'longitude',
    'Easting',
    'Northing',
    'No. of Cas',
    'Casualty_1',
    'Casualty A',
    'junction_detail_historic',
    'junction_detail',
    'junction_control',
    'second_road_class',
    'first_road_number',
    'second_road_number',
    'Borough Na',
    '_Casualt_1',
    '_Casualt_3',
    '_Collisi_1',
    'lsoa_of_ac'
]
MAPPINGS_PATH = "categorical_mappings.json"
merged_path = 'merged_collision_data_speed_limit.csv'

# ----------------------------
# 1) Read merged CSV
# ----------------------------
if not os.path.exists(merged_path):
    raise FileNotFoundError(f"{merged_path} not found in current directory.")

merged_df = pd.read_csv(merged_path, low_memory=False)
print(f"Loaded merged file -> shape: {merged_df.shape}")

# Drop specified columns if present
cols_present = [c for c in columns_to_discard if c in merged_df.columns]
if cols_present:
    merged_df = merged_df.drop(columns=cols_present)
    print(f"Dropped columns_to_discard: {cols_present}")

# ----------------------------
# 2) Normalize target to 'Severity'
# ----------------------------
if 'Severity' not in merged_df.columns and 'collision_severity' in merged_df.columns:
    merged_df['Severity'] = merged_df['collision_severity']
if 'collision_severity' in merged_df.columns:
    merged_df = merged_df.drop(columns=['collision_severity'])

merged_df = merged_df.dropna(subset=['Severity'])
merged_df['Severity'] = merged_df['Severity'].astype(str).str.upper().str.strip()

print("\nTarget distribution (raw):")
print(merged_df['Severity'].value_counts())

# ----------------------------
# 3) Optional: extract hour/month if date/time exist
# ----------------------------
if 'date' in merged_df.columns and 'time' in merged_df.columns:
    merged_df['date'] = pd.to_datetime(merged_df['date'], dayfirst=True, errors='coerce')
    merged_df['time'] = pd.to_datetime(merged_df['time'], format='%H:%M:%S', errors='coerce')
    merged_df['hour_of_day'] = merged_df['time'].dt.hour
    merged_df['month'] = merged_df['date'].dt.month
    merged_df = merged_df.drop(columns=['date', 'time'])
    print("Extracted 'hour_of_day' and 'month'")

# ----------------------------
# 4) Build X_df and detect numeric/categorical
# ----------------------------
X_df = merged_df.drop(columns=['Severity']).copy()

# Coerce obvious numeric strings to numeric
for col in X_df.columns:
    if X_df[col].dtype == 'object' or pd.api.types.is_string_dtype(X_df[col]):
        non_null = X_df[col].dropna().astype(str)
        if len(non_null) > 0:
            parsed = non_null.str.replace(r'[^\d\.\-]', '', regex=True).replace('', np.nan)
            try:
                frac_numeric = parsed.notna().sum() / len(non_null)
            except Exception:
                frac_numeric = 0.0
            if frac_numeric > 0.9:
                X_df[col] = pd.to_numeric(X_df[col], errors='coerce')

numeric_cols = X_df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_df.select_dtypes(exclude=['number']).columns.tolist()

# Drop numeric columns containing any NaN (no numeric imputation)
numeric_cols_with_nan = [c for c in numeric_cols if X_df[c].isnull().any()]
if numeric_cols_with_nan:
    print(f"Dropping numeric columns that contain NaN (no imputations): {numeric_cols_with_nan}")
    X_df = X_df.drop(columns=numeric_cols_with_nan)
    numeric_cols = X_df.select_dtypes(include=['number']).columns.tolist()
    categorical_cols = X_df.select_dtypes(exclude=['number']).columns.tolist()

print(f"\nRemaining numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Remaining categorical columns ({len(categorical_cols)}): {categorical_cols}")

if len(numeric_cols) + len(categorical_cols) == 0:
    raise RuntimeError("No features remain after dropping numeric columns containing NaN and discarding others.")

# ----------------------------
# 5) Encode target and split (we split BEFORE creating mappings so mappings come from trainval)
# ----------------------------
# ----------------------------
# 5) Encode target as binary and split (SEVERE vs SLIGHT)
# ----------------------------
y_series = merged_df['Severity'].reset_index(drop=True)

# Binary mapping: 0 = SLIGHT, 1 = SEVERE (SERIOUS + FATAL)
y_binary = (y_series != 'SLIGHT').astype(int).values
binary_classes = np.array(['SLIGHT', 'SEVERE'])

X_trainval_df, X_test_df, y_trainval, y_test = train_test_split(
    X_df, y_binary, test_size=0.05, stratify=y_binary, random_state=SEED
)

# adjust n_splits based on smallest class in trainval
counts = np.bincount(y_trainval)
min_count = counts.min()
n_splits = 5
if min_count < n_splits:
    n_splits = int(min_count)
    if n_splits < 2:
        raise RuntimeError("Not enough samples per class to perform CV. Need at least 2 samples in smallest class.")
    print(f"Reduced StratifiedKFold n_splits to {n_splits} because smallest class has {min_count} samples.")

num_classes = 2
print(f"\nTarget classes: {list(binary_classes)} (num_classes={num_classes})")


# ----------------------------
# 6) Load saved mappings (if exists) and CREATE missing mappings from trainval, then APPLY to trainval & test
# ----------------------------
if os.path.exists(MAPPINGS_PATH):
    with open(MAPPINGS_PATH, 'r', encoding='utf-8') as fh:
        saved_mappings = json.load(fh)
        if not isinstance(saved_mappings, dict):
            saved_mappings = {}
else:
    saved_mappings = {}

# Ensure all categorical columns exist in saved_mappings; if missing, create from X_trainval_df
for col in categorical_cols:
    if col not in saved_mappings:
        print(f"Mapping for '{col}' missing. Creating mapping from train+val data and adding to {MAPPINGS_PATH}.")
        uniques = pd.Series(X_trainval_df[col].astype(str).fillna("")).unique().tolist()
        saved_mappings[col] = {str(val): idx + 1 for idx, val in enumerate(uniques)}

# Apply mapping to trainval and test (NaN or unseen -> 0)
def apply_mapping_df(df, mappings, cat_cols):
    out = df.copy()
    for col in cat_cols:
        md = mappings[col]
        out[col] = out[col].map(lambda x: int(md.get(str(x), 0)) if pd.notna(x) else 0).astype(int)
    return out

X_trainval_df = apply_mapping_df(X_trainval_df, saved_mappings, categorical_cols)
X_test_df = apply_mapping_df(X_test_df, saved_mappings, categorical_cols)

# Save updated mappings back to file
with open(MAPPINGS_PATH, 'w', encoding='utf-8') as fh:
    json.dump(saved_mappings, fh, ensure_ascii=False, indent=4)

# After mapping, categorical columns are integer-coded; feature order:
feature_names = numeric_cols + categorical_cols
print("All columns left in the data set:", feature_names)
print(f"\nAfter mappings -> numeric cols: {len(numeric_cols)}, categorical cols (coded): {len(categorical_cols)}")

# ----------------------------
# 7) Helpers for building arrays (we will use the global mappings already applied)
# ----------------------------
def build_numeric_arrays(df, numeric_cols):
    if not numeric_cols:
        return np.zeros((len(df), 0), dtype=np.float32)
    return df[numeric_cols].astype(float).to_numpy()

def build_categorical_arrays(df, categorical_cols):
    if not categorical_cols:
        return np.zeros((len(df), 0), dtype=np.float32)
    return df[categorical_cols].astype(float).to_numpy()

# ----------------------------
# 8) Model factory
# ----------------------------
def create_model(input_dim, num_classes):
    inputs = keras.Input(shape=(input_dim,))
    x = keras.layers.Dense(128, activation='relu')(inputs)
    x = keras.layers.Dropout(0.3)(x)
    x = keras.layers.Dense(64, activation='relu')(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(num_classes, activation='softmax')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# ----------------------------
# Feature importance config & helper
# ----------------------------
FEATURE_IMPORTANCE_THRESHOLD = 1e-3  # mean drop in macro recall below which a feature is considered non-relevant
FEATURE_IMPORTANCE_TOP_K = None       # if set to an integer, keep top K features instead of threshold
N_REPEATS_IMPORTANCE = 10


def compute_permutation_importance_keras(model, X, y, feature_names, n_repeats=10, random_state=42):
    """
    Compute permutation importance for a keras classifier using macro recall as metric.
    Returns array of importance values (baseline_recall - permuted_recall) for each feature.
    """
    rng = np.random.RandomState(random_state)
    baseline_prob = model.predict(X, verbose=0)
    baseline_pred = np.argmax(baseline_prob, axis=1)
    # use macro recall as a stable metric for imbalance
    baseline_score = recall_score(y, baseline_pred, average='macro')

    importances = np.zeros(len(feature_names), dtype=float)

    for fi in range(len(feature_names)):
        drops = []
        col = X[:, fi].copy()
        for r in range(n_repeats):
            perm = col.copy()
            rng.shuffle(perm)
            X_perm = X.copy()
            X_perm[:, fi] = perm
            y_perm = np.argmax(model.predict(X_perm, verbose=0), axis=1)
            perm_score = recall_score(y, y_perm, average='macro')
            drops.append(baseline_score - perm_score)
        importances[fi] = np.mean(drops)
    return importances

# ----------------------------
# 9) Stratified CV using global mappings (created from trainval)
# ----------------------------
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

all_acc = []
all_loss = []
all_bal_acc = []  # balanced accuracy across folds (validation)
fold_importances = []
last_history = None
last_model = None
last_X_val = None
last_y_val = None

for fold, (train_idx, val_idx) in enumerate(skf.split(X_trainval_df, y_trainval), start=1):
    print(f"\n--- Fold {fold} ---")
    X_train_df = X_trainval_df.iloc[train_idx].reset_index(drop=True)
    X_val_df   = X_trainval_df.iloc[val_idx].reset_index(drop=True)
    y_train = y_trainval[train_idx]
    y_val   = y_trainval[val_idx]

    # Build arrays
    X_train_num = build_numeric_arrays(X_train_df, numeric_cols)
    X_val_num = build_numeric_arrays(X_val_df, numeric_cols)

    # scale numeric per-fold
    if X_train_num.shape[1] > 0:
        means = np.nanmean(X_train_num, axis=0)
        stds = np.nanstd(X_train_num, axis=0)
        stds[stds == 0] = 1.0
        X_train_num = ((X_train_num - means) / stds).astype(np.float32)
        X_val_num = ((X_val_num - means) / stds).astype(np.float32)
    else:
        X_train_num = X_train_num.astype(np.float32)
        X_val_num = X_val_num.astype(np.float32)

    X_train_cat = build_categorical_arrays(X_train_df, categorical_cols)
    X_val_cat = build_categorical_arrays(X_val_df, categorical_cols)

    X_train = np.hstack([X_train_num, X_train_cat]).astype(np.float32)
    X_val = np.hstack([X_val_num, X_val_cat]).astype(np.float32)

    print(f"Fold shapes -> train: {X_train.shape}, val: {X_val.shape}; classes in train: {np.unique(y_train)}")

    model = create_model(X_train.shape[1], num_classes)

    # class weights mapped to actual labels
    classes_in_fold = np.unique(y_train)
    cw = class_weight.compute_class_weight(class_weight='balanced', classes=classes_in_fold, y=y_train)
    cw_dict = {int(c): float(w) for c, w in zip(classes_in_fold, cw)}


    es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=64,
        class_weight=cw_dict,
        callbacks=[es],
        verbose=1
    )

    loss, acc = model.evaluate(X_val, y_val, verbose=0)
    # compute balanced accuracy on the validation set for this fold
    y_val_pred = np.argmax(model.predict(X_val), axis=1)
    bal_acc = balanced_accuracy_score(y_val, y_val_pred)

    print(f"Fold {fold} -> val_loss: {loss:.4f}, val_acc: {acc:.4f}, val_balanced_acc: {bal_acc:.4f}")
    all_acc.append(acc)
    all_loss.append(loss)
    all_bal_acc.append(bal_acc)

    # Compute permutation importance on this fold's validation set and store (no test leakage)
    print(f"Computing permutation importance for fold {fold} (n_repeats={N_REPEATS_IMPORTANCE})...")
    fi_vals = compute_permutation_importance_keras(model, X_val, y_val, feature_names, n_repeats=N_REPEATS_IMPORTANCE, random_state=SEED+fold)
    fold_importances.append(fi_vals)

    last_history = history
    last_model = model
    last_X_val = X_val
    last_y_val = y_val

# ----------------------------
# 10) CV summary + plots
# ----------------------------
print(f"\nAverage Validation Accuracy: {np.mean(all_acc):.4f} (+/- {np.std(all_acc):.4f})")
print(f"Average Validation Loss: {np.mean(all_loss):.4f} (+/- {np.std(all_loss):.4f})")

if last_history is not None:
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.plot(last_history.history.get('accuracy', []), label='train_acc')
    plt.plot(last_history.history.get('val_accuracy', []), label='val_acc')
    plt.legend(); plt.title('Last fold acc')
    plt.subplot(1,2,2)
    plt.plot(last_history.history.get('loss', []), label='train_loss')
    plt.plot(last_history.history.get('val_loss', []), label='val_loss')
    plt.legend(); plt.title('Last fold loss')
    plt.tight_layout(); plt.show()

if last_model is not None and last_X_val is not None:
    y_pred_prob = last_model.predict(last_X_val)
    y_pred = np.argmax(y_pred_prob, axis=1)

    unique_labels_in_fold = np.unique(last_y_val)
    target_names_in_fold = binary_classes[unique_labels_in_fold]


    print("\nClassification report (last fold):")
    print(classification_report(last_y_val, y_pred, labels=unique_labels_in_fold, target_names=target_names_in_fold))

    cm = confusion_matrix(last_y_val, y_pred)
    plt.figure(figsize=(7,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names_in_fold, yticklabels=target_names_in_fold)
    plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix (last fold)'); plt.show()

    # Permutation importance (last fold) -- kept for quick inspection
    print("\nRunning permutation importance (this may take a bit)...")
    baseline_recall = recall_score(last_y_val, y_pred, labels=unique_labels_in_fold, average=None)
    n_repeats = 10
    rng = np.random.RandomState(42)

    feat_importance = np.zeros((len(feature_names), len(unique_labels_in_fold)), dtype=float)

    for fi in range(len(feature_names)):
        drops = np.zeros((n_repeats, len(unique_labels_in_fold)), dtype=float)
        col = last_X_val[:, fi].copy()
        for r in range(n_repeats):
            perm = col.copy()
            rng.shuffle(perm)
            X_perm = last_X_val.copy()
            X_perm[:, fi] = perm
            y_perm_prob = last_model.predict(X_perm, verbose=0)
            y_perm = np.argmax(y_perm_prob, axis=1)
            perm_recall = recall_score(last_y_val, y_perm, labels=unique_labels_in_fold, average=None)
            drops[r, :] = baseline_recall - perm_recall
        feat_importance[fi, :] = drops.mean(axis=0)

    top_k = 10
    for ci, lbl in enumerate(unique_labels_in_fold):
        cls_name = binary_classes[int(lbl)]
        importance_for_class = feat_importance[:, ci]
        sorted_idx = np.argsort(-importance_for_class)
        top_idx = sorted_idx[:top_k]
        print(f"\nTop {min(top_k, len(feature_names))} features for class '{cls_name}' (mean recall drop):")
        for rank, idx in enumerate(top_idx, start=1):
            fname = feature_names[idx]
            val = importance_for_class[idx]
            print(f" {rank:2d}. {fname:40s}  drop={val:.4f}")

        plt.figure(figsize=(8,4))
        names = [feature_names[i] for i in top_idx]
        vals = importance_for_class[top_idx]
        plt.barh(range(len(vals))[::-1], vals)
        plt.yticks(range(len(vals))[::-1], names)
        plt.xlabel('Mean drop in recall after permutation')
        plt.title(f"Top features for class '{cls_name}'")
        plt.tight_layout()
        plt.show()

# ----------------------------
# 11) Final model trained on full trainval and evaluated on test (no leakage)
# ----------------------------
# Build final trainval and test arrays using the SAME mappings (already applied)
# Numeric scaling using full trainval
X_trainval_num = build_numeric_arrays(X_trainval_df, numeric_cols)
X_test_num = build_numeric_arrays(X_test_df, numeric_cols)

if X_trainval_num.shape[1] > 0:
    final_means = np.nanmean(X_trainval_num, axis=0)
    final_stds = np.nanstd(X_trainval_num, axis=0)
    final_stds[final_stds == 0] = 1.0
    X_trainval_num = ((X_trainval_num - final_means) / final_stds).astype(np.float32)
    X_test_num = ((X_test_num - final_means) / final_stds).astype(np.float32)

X_trainval_cat = build_categorical_arrays(X_trainval_df, categorical_cols)
X_test_cat = build_categorical_arrays(X_test_df, categorical_cols)

X_trainval_final = np.hstack([X_trainval_num, X_trainval_cat]).astype(np.float32)
X_test_final = np.hstack([X_test_num, X_test_cat]).astype(np.float32)

classes_full = np.unique(y_trainval)
cw_full = class_weight.compute_class_weight(class_weight='balanced', classes=classes_full, y=y_trainval)
cw_full_dict = {int(c): float(w) for c, w in zip(classes_full, cw_full)}

final_model = create_model(X_trainval_final.shape[1], num_classes)
es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
final_model.fit(X_trainval_final, y_trainval, validation_split=0.1, epochs=100, batch_size=64,
                class_weight=cw_full_dict, callbacks=[es], verbose=1)

y_test_pred = np.argmax(final_model.predict(X_test_final), axis=1)

print("\n--- FINAL TEST SET EVALUATION (UNSEEN DATA) ---")
print(classification_report(y_test, y_test_pred, target_names=binary_classes))

# ----------------------------
# 12) Aggregate CV permutation importance and automatic feature dropping (no test leakage)
# ----------------------------
if len(fold_importances) == 0:
    raise RuntimeError("No fold importances were collected during CV — cannot perform CV-based feature selection.")

# Average importances across folds
mean_importances = np.mean(np.vstack(fold_importances), axis=0)

# present sorted importance (CV-averaged)
sorted_idx = np.argsort(-mean_importances)
plt.figure(figsize=(10, max(4, len(feature_names)*0.25)))
names_sorted = [feature_names[i] for i in sorted_idx]
vals_sorted = mean_importances[sorted_idx]
plt.barh(range(len(vals_sorted))[::-1], vals_sorted)
plt.yticks(range(len(vals_sorted))[::-1], names_sorted)
plt.xlabel('Mean drop in macro-recall after permutation (CV avg)')
plt.title('CV-averaged permutation importance')
plt.tight_layout(); plt.show()

for rank, idx in enumerate(sorted_idx[:30], start=1):
    print(f"{rank:2d}. {feature_names[idx]:40s} -> importance={mean_importances[idx]:.6f}")

# Decide which features to keep (based on CV average)
if FEATURE_IMPORTANCE_TOP_K is not None:
    keep_idx = sorted_idx[:FEATURE_IMPORTANCE_TOP_K]
else:
    keep_idx = np.where(mean_importances > FEATURE_IMPORTANCE_THRESHOLD)[0]

# If thresholding removed all features, fall back to keeping the top 5
if len(keep_idx) == 0:
    fallback_k = min(5, len(feature_names))
    print(f"\nWarning: no features passed the importance threshold {FEATURE_IMPORTANCE_THRESHOLD}. Keeping top {fallback_k} features instead.")
    keep_idx = sorted_idx[:fallback_k]

keep_idx = np.sort(keep_idx)
kept_feature_names = [feature_names[i] for i in keep_idx]
removed_feature_names = [f for f in feature_names if f not in kept_feature_names]

print(f"\nKept {len(kept_feature_names)} features (CV-based): {kept_feature_names}")
print(f"Removed {len(removed_feature_names)} features: {removed_feature_names}")

# Update dataframes and mappings to drop removed features
X_trainval_df_reduced = X_trainval_df[kept_feature_names].copy()
X_test_df_reduced = X_test_df[kept_feature_names].copy()

# Update numeric/categorical lists
numeric_cols_reduced = [c for c in numeric_cols if c in kept_feature_names]
categorical_cols_reduced = [c for c in categorical_cols if c in kept_feature_names]

# Remove dropped categorical mappings and save mappings
for dropped in [c for c in categorical_cols if c not in categorical_cols_reduced]:
    if dropped in saved_mappings:
        print(f"Removing mapping for dropped categorical column: {dropped}")
        saved_mappings.pop(dropped, None)

with open(MAPPINGS_PATH, 'w', encoding='utf-8') as fh:
    json.dump(saved_mappings, fh, ensure_ascii=False, indent=4)

# Rebuild arrays using reduced features and rescale numeric using trainval
X_trainval_num_r = build_numeric_arrays(X_trainval_df_reduced, numeric_cols_reduced)
X_test_num_r = build_numeric_arrays(X_test_df_reduced, numeric_cols_reduced)

if X_trainval_num_r.shape[1] > 0:
    final_means_r = np.nanmean(X_trainval_num_r, axis=0)
    final_stds_r = np.nanstd(X_trainval_num_r, axis=0)
    final_stds_r[final_stds_r == 0] = 1.0
    X_trainval_num_r = ((X_trainval_num_r - final_means_r) / final_stds_r).astype(np.float32)
    X_test_num_r = ((X_test_num_r - final_means_r) / final_stds_r).astype(np.float32)
else:
    X_trainval_num_r = np.zeros((len(X_trainval_df_reduced), 0), dtype=np.float32)
    X_test_num_r = np.zeros((len(X_test_df_reduced), 0), dtype=np.float32)

X_trainval_cat_r = build_categorical_arrays(X_trainval_df_reduced, categorical_cols_reduced)
X_test_cat_r = build_categorical_arrays(X_test_df_reduced, categorical_cols_reduced)

X_trainval_final_r = np.hstack([X_trainval_num_r, X_trainval_cat_r]).astype(np.float32)
X_test_final_r = np.hstack([X_test_num_r, X_test_cat_r]).astype(np.float32)

print(f"\nShapes after feature removal -> trainval: {X_trainval_final_r.shape}, test: {X_test_final_r.shape}")

# Retrain model on reduced feature set and evaluate
if X_trainval_final_r.shape[1] == 0:
    raise RuntimeError("No features left after reduction. Aborting retrain.")

final_model_reduced = create_model(X_trainval_final_r.shape[1], num_classes)
final_model_reduced.fit(X_trainval_final_r, y_trainval, validation_split=0.1, epochs=100, batch_size=64,
                class_weight=cw_full_dict, callbacks=[es], verbose=1)

y_test_pred_r = np.argmax(final_model_reduced.predict(X_test_final_r), axis=1)

print("\n--- FINAL TEST SET EVALUATION AFTER FEATURE REDUCTION (CV-based) ---")
print(classification_report(y_test, y_test_pred_r, target_names=binary_classes))

# Save a lightweight report of CV-averaged importances
importance_report_cv = {name: float(imp) for name, imp in zip(feature_names, mean_importances)}
with open('feature_importances_cv_report.json', 'w', encoding='utf-8') as fh:
    json.dump(importance_report_cv, fh, ensure_ascii=False, indent=4)

print('\nSaved feature_importances_cv_report.json with CV-averaged importance values.')

# Compute confusion matrix
cm = confusion_matrix(y_test, y_test_pred_r)

# Print raw counts
print("Confusion matrix (counts):")
print(cm)

# Plot confusion matrix
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=binary_classes, yticklabels=binary_classes)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion matrix — final test set')
plt.show()

In [ ]:
# pearson_for_model_inputs.py
import os, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# try to use scipy for p-values
try:
    from scipy.stats import pearsonr
    SCIPY = True
except Exception:
    SCIPY = False

# ----------------------------
# Config (uses your original lists/paths)
# ----------------------------
SEED = 42
merged_path = "merged_collision_data_speed_limit.csv"
MAPPINGS_PATH = "categorical_mappings.json"
OUT_CSV = "feature_correlations_kept_features_auto.csv"
TOP_N = 30        # features in bar plot
HEATMAP_K = 20    # features in heatmap

columns_to_discard = [
    'collision_index',
    'collision_ref_no',
    'location_easting_osgr',
    'location_northing_osgr',
    'local_authority_ons_district',
    'local_authority_highway',
    'local_authority_highway_current',
    'lsoa_of_accident_location',
    'AREFNO',
    'collision_severity',
    'enhanced_severity_collision',
    'collision_injury_based',
    'collision_adjusted_severity_serious',
    'collision_adjusted_severity_slight',
    'CREFNO',
    'latitude',
    'longitude',
    'Easting',
    'Northing',
    'No. of Cas',
    'Casualty_1',
    'Casualty A',
    'junction_detail_historic',
    'junction_detail',
    'junction_control',
    'second_road_class',
    'first_road_number',
    'second_road_number',
    'Borough Na',
    '_Casualt_1',
    '_Casualt_3',
    '_Collisi_1',
    'lsoa_of_ac'
]

# ----------------------------
# 1) Load data & drop discard columns
# ----------------------------
if not os.path.exists(merged_path):
    raise FileNotFoundError(f"{merged_path} not found in current directory.")

df = pd.read_csv(merged_path, low_memory=False)
print(f"Loaded {merged_path} shape: {df.shape}")

cols_present = [c for c in columns_to_discard if c in df.columns]
if cols_present:
    df = df.drop(columns=cols_present)
    print(f"Dropped columns_to_discard: {cols_present}")

# ----------------------------
# 2) Normalize Severity target
# ----------------------------
if 'Severity' not in df.columns and 'collision_severity' in df.columns:
    df['Severity'] = df['collision_severity']
if 'collision_severity' in df.columns:
    df = df.drop(columns=['collision_severity'])

df = df.dropna(subset=['Severity'])
df['Severity'] = df['Severity'].astype(str).str.upper().str.strip()
print("\nTarget distribution (raw):")
print(df['Severity'].value_counts())

# ----------------------------
# 3) Optional: extract hour/month if date/time exist (like your pipeline)
# ----------------------------
if 'date' in df.columns and 'time' in df.columns:
    df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
    df['time'] = pd.to_datetime(df['time'], format='%H:%M:%S', errors='coerce')
    df['hour_of_day'] = df['time'].dt.hour
    df['month'] = df['date'].dt.month
    df = df.drop(columns=['date', 'time'])
    print("Extracted 'hour_of_day' and 'month'")

# ----------------------------
# 4) Build X_df and detect numeric/categorical (same heuristics)
# ----------------------------
X_df = df.drop(columns=['Severity']).copy()

# Coerce obvious numeric strings to numeric (same heuristic)
for col in X_df.columns:
    if X_df[col].dtype == 'object' or pd.api.types.is_string_dtype(X_df[col]):
        non_null = X_df[col].dropna().astype(str)
        if len(non_null) > 0:
            parsed = non_null.str.replace(r'[^\d\.\-]', '', regex=True).replace('', np.nan)
            try:
                frac_numeric = parsed.notna().sum() / len(non_null)
            except Exception:
                frac_numeric = 0.0
            if frac_numeric > 0.9:
                X_df[col] = pd.to_numeric(X_df[col], errors='coerce')

numeric_cols = X_df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_df.select_dtypes(exclude=['number']).columns.tolist()

# Drop numeric columns containing any NaN (no numeric imputation) - matches your pipeline
numeric_cols_with_nan = [c for c in numeric_cols if X_df[c].isnull().any()]
if numeric_cols_with_nan:
    print(f"Dropping numeric columns that contain NaN (no imputations): {numeric_cols_with_nan}")
    X_df = X_df.drop(columns=numeric_cols_with_nan)
    numeric_cols = X_df.select_dtypes(include=['number']).columns.tolist()
    categorical_cols = X_df.select_dtypes(exclude=['number']).columns.tolist()

print(f"\nRemaining numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Remaining categorical columns ({len(categorical_cols)}): {categorical_cols}")

if len(numeric_cols) + len(categorical_cols) == 0:
    raise RuntimeError("No features remain after dropping numeric columns containing NaN and discarding others.")

# ----------------------------
# 5) Encode target and split (we split BEFORE creating mappings so mappings come from trainval)
# ----------------------------
y_series = df['Severity'].reset_index(drop=True)
y_binary = (y_series != 'SLIGHT').astype(int).values
binary_classes = np.array(['SLIGHT', 'SEVERE'])

X_trainval_df, X_test_df, y_trainval, y_test = train_test_split(
    X_df, y_binary, test_size=0.05, stratify=y_binary, random_state=SEED
)
print(f"\nTrain+val shape: {X_trainval_df.shape} Test shape: {X_test_df.shape}")

# ----------------------------
# 6) Load saved mappings (if exists) and CREATE missing mappings from trainval, then APPLY to trainval & test
# ----------------------------
if os.path.exists(MAPPINGS_PATH):
    with open(MAPPINGS_PATH, 'r', encoding='utf-8') as fh:
        saved_mappings = json.load(fh)
        if not isinstance(saved_mappings, dict):
            saved_mappings = {}
else:
    saved_mappings = {}

# Create missing mappings from train+val (same as your pipeline)
for col in categorical_cols:
    if col not in saved_mappings:
        print(f"Mapping for '{col}' missing. Creating mapping from train+val data.")
        uniques = pd.Series(X_trainval_df[col].astype(str).fillna("")).unique().tolist()
        saved_mappings[col] = {str(val): idx + 1 for idx, val in enumerate(uniques)}

# Apply mapping to trainval and test (NaN or unseen -> 0)
def apply_mapping_df(df_in, mappings, cat_cols):
    out = df_in.copy()
    for col in cat_cols:
        md = mappings[col]
        out[col] = out[col].map(lambda x: int(md.get(str(x), 0)) if pd.notna(x) else 0).astype(int)
    return out

X_trainval_mapped = apply_mapping_df(X_trainval_df, saved_mappings, categorical_cols)
# (we don't need X_test_mapped for correlations, but we mirror your pipeline)
X_test_mapped = apply_mapping_df(X_test_df, saved_mappings, categorical_cols)

# Save updated mappings back to file (same behaviour)
with open(MAPPINGS_PATH, 'w', encoding='utf-8') as fh:
    json.dump(saved_mappings, fh, ensure_ascii=False, indent=4)

# ----------------------------
# 7) Features that will be fed to the model (after mapping)
# ----------------------------
feature_names = numeric_cols + categorical_cols
print("\nAll columns left in the data set (to be fed to model):", feature_names)

# Take only those columns from the mapped trainval (these are the inputs to the model)
X_kept_trainval = X_trainval_mapped[feature_names].reset_index(drop=True)
y_kept_trainval = pd.Series(y_trainval).reset_index(drop=True)

# Ensure numeric dtype (numeric cols and mapped categoricals are ints)
X_kept_trainval = X_kept_trainval.select_dtypes(include=[np.number])
print(f"\nColumns used for Pearson (count={len(X_kept_trainval.columns)}): {X_kept_trainval.columns.tolist()}")

# ----------------------------
# 8) Compute Pearson correlation (point-biserial) on train+val
# ----------------------------
results = []
for col in X_kept_trainval.columns:
    mask = X_kept_trainval[col].notna() & y_kept_trainval.notna()
    n = int(mask.sum())
    if n < 3:
        r = float("nan"); p = float("nan")
    else:
        xvals = X_kept_trainval.loc[mask, col].astype(float).values
        yvals = y_kept_trainval.loc[mask].astype(int).values
        if SCIPY:
            try:
                r, p = pearsonr(xvals, yvals)
            except Exception:
                r = float(np.corrcoef(xvals, yvals)[0,1]); p = float("nan")
        else:
            r = float(np.corrcoef(xvals, yvals)[0,1]); p = float("nan")
    results.append({"feature": col, "pearson_r": r, "abs_r": abs(r) if not math.isnan(r) else np.nan, "p_value": p, "n": n})

corr_df = pd.DataFrame(results).dropna(subset=["pearson_r"]).sort_values("abs_r", ascending=False).reset_index(drop=True)
corr_df.to_csv(OUT_CSV, index=False)
print(f"\nSaved correlations to: {OUT_CSV}")

# ----------------------------
# 9) Print top-10 and plot bar chart + heatmap
# ----------------------------
print("\nTop 10 features by |Pearson r|:")
print(corr_df[["feature", "pearson_r", "p_value", "n"]].head(10).to_string(index=False))

# Bar chart
topn = min(TOP_N, len(corr_df))
top_df = corr_df.head(topn)
plt.figure(figsize=(10, max(4, topn * 0.28)))
plt.barh(top_df["feature"][::-1], top_df["pearson_r"][::-1])
plt.xlabel("Pearson r (with binary target 0=SLIGHT,1=SEVERE)")
plt.title(f"Top {topn} features by |Pearson r| (kept inputs)")
for i, v in enumerate(top_df["pearson_r"].values[::-1]):
    plt.text(v, i, f" {v:.3f}", va="center", fontsize=8)
plt.tight_layout()
plt.show()

# # Heatmap for top-K features
# k = min(HEATMAP_K, len(corr_df))
# top_feats = corr_df["feature"].head(k).tolist()
# if len(top_feats) > 1:
#     cm = X_kept_trainval[top_feats].corr(method="pearson")
#     plt.figure(figsize=(min(14, k*0.6), min(12, k*0.6)))
#     sns.heatmap(cm, annot=True, fmt=".2f", cmap="coolwarm", cbar_kws={"shrink":0.7})
#     plt.title(f"Feature-feature Pearson correlation (top {k} kept features by |r|)")
#     plt.tight_layout()
#     plt.show()
# else:
#     print("Not enough kept features to plot heatmap.")

# ----------------------------
# Final cautions
# ----------------------------
print("\nCaveats:")
print("- This computed Pearson correlations on train+val only using the features AS THEY WILL BE FED to the model (numeric columns + integer-coded categoricals).")
print("- Integer codes for categorical features are arbitrary labels (even though consistent across train+val) — Pearson can be misleading for non-ordinal categories.")
if not SCIPY:
    print("- SciPy not installed: p-values are NaN. Install scipy to compute p-values via scipy.stats.pearsonr.")
